# Linear Regression

In this exercise, we introduce linear regression and explore how it can be used to _predict the band gap of oxide materials_.

By the end of this exercise, users should be able to:

- Load a dataset into a pandas DataFrame.
- Train a linear regression model.
- Evaluate model performance using regression metrics.
- Use cross validation to assess consistency of model.

## Dependencies and environment

Everything used in this notebook is already available in the environment, so there is nothing to install.


In [ ]:
import os
import warnings
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_validate

plt.rcParams["figure.figsize"] = (6, 4)
pd.set_option("display.max_columns", 12)


## Download and load the dataset

The next cell downloads the dataset and reads it into a pandas DataFrame.

In [ ]:
# data setup configuration
data_path = 'ai-ml-course'
full_dir = os.path.join(data_path, 'Material')
os.makedirs(full_dir, exist_ok=True)

base_url = "https://virgodb.cosma.dur.ac.uk/public/diractraining/ai-ml-course/Material/"
files_to_download = ["oxides.pickle"]

for file_name in files_to_download:
    local_file = os.path.join(full_dir, file_name)
    if os.path.exists(local_file):
        print(f"{file_name} is already available.")
    else:
        print(f"Downloading {file_name} ...")
        urllib.request.urlretrieve(base_url + file_name, local_file)
        print("Done.")

In [ ]:
#Load the dataset

oxides = pd.read_pickle(os.path.join(full_dir, "oxides.pickle"))

print("Read dataset completed successfully.")
print("Total number of rows:   {0}".format(len(oxides.index)))
print("Total number of columns: {0}".format(len(oxides.columns)))

## Aliases

The original column names are long. We rename the columns we use most often to short **aliases** and tidy the remaining names so they are easier to type.


In [ ]:
#Create aliases

# Short aliases for the columns we will use most.
aliases = {
    "gllbsc_gap": "band_gap",
    "MagpieData avg_dev Electronegativity": "electroneg_dev",
    "MagpieData avg_dev CovalentRadius": "cov_radius_dev",
    "MagpieData maximum NsValence": "max_ns_valence",
    "MagpieData minimum Number": "min_atomic_num",
    "MagpieData maximum Number": "max_atomic_num",
}

# Every other column: lower-case, no spaces.
def tidy(name):
    return aliases.get(name, name.replace("MagpieData ", "").replace(" ", "_").lower())

oxides = oxides.rename(columns=tidy)

LABEL = "band_gap"
all_features = [c for c in oxides.columns if c != LABEL]
print(f"{len(all_features)} descriptor columns and 1 label ({LABEL}).")
print("First few descriptor names:", all_features[:6])

## Split the dataset

We keep 20% of the materials aside as a **test set**. The model does not see these examples during training or hyperparameter selection, so we can use them for a final check of how well it predicts unseen materials.


In [ ]:
#Split the dataset

training_df, test_df = train_test_split(oxides, test_size=0.2, random_state=0)

print("Training examples:", len(training_df))
print("Test examples:    ", len(test_df))
training_df[[LABEL, "electroneg_dev", "cov_radius_dev", "max_ns_valence"]].head(10)

## View dataset statistics

In this cell we will explore how our data looks like. Here you will use `DataFrame.describe` to see descriptive statistics for the label and a few descriptors.
    
Questions:
  * What is the maximum band gap, and what is the mean band gap?
  * How many materials have a band gap of exactly zero?
  * Are any features missing data?

In [ ]:
#View dataset statistics

training_df[[LABEL, "electroneg_dev", "cov_radius_dev", "max_ns_valence"]].describe()

<details>



<summary>Show solutions</summary>

```python
print("Maximum band gap: {:.2f} eV".format(training_df[LABEL].max()))
print("Mean band gap:    {:.2f} eV".format(training_df[LABEL].mean()))
print("Materials with band gap == 0 (metals): {}".format((training_df[LABEL] == 0).sum()))
print("Missing values in the whole table:      {}".format(int(training_df.isnull().sum().sum())))
```

</details>

## Generate a correlation matrix

Which features are most related to the label? A **correlation matrix** helps answer this:

- **`1.0`**: perfect positive correlation (when one rises, the other rises).
- **`-1.0`**: perfect negative correlation (when one rises, the other falls).
- **`0.0`**: no *linear* relationship.

The larger the absolute value, the more useful the feature is likely to be for a linear model.

Question:
- Which of these features correlates most strongly with `band_gap`?

In [ ]:
#View correlation matrix

selected = [LABEL, "electroneg_dev", "cov_radius_dev", "max_ns_valence", "min_atomic_num", "max_atomic_num"]
training_df[selected].corr().round(2)

<details>
<summary>Show Solution</summary>

```python
corr = training_df[selected].corr()[LABEL].drop(LABEL)
print(corr.round(3).to_string())
print("\nStrongest correlation among these features:", corr.abs().idxmax())
print("Weakest correlation among these features:  ", corr.abs().idxmin())
   ```
</details>

In [ ]:
#View scatter plots

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, feature in zip(axes, ["electroneg_dev", "cov_radius_dev", "max_ns_valence"]):
    ax.scatter(training_df[feature], training_df[LABEL], s=5, alpha=0.3)
    ax.set_xlabel(feature)
axes[0].set_ylabel("band_gap (eV)")
plt.tight_layout()
plt.show()

## Define functions

A linear regression model has the form

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + b$$

where the $x$ values are the **features**, the $w$ values are the **weights**, and $b$ is the **bias**. Training finds weights and a bias that make predictions $\hat{y}$ close to the observed labels $y$.

**Metrics** used to compare runs:

- **MAE** (mean absolute error): the average absolute prediction error, in eV.
- **RMSE** (root mean squared error): similar to MAE, but penalises large errors more strongly.
- **R²**: a measure of how much variation in the label is explained by the model; 1.0 is a perfect fit.

During the initial experiments, we will compare training performance and inspect predictions on the training data. We will use cross validation on the training set to assess consistency, and keep the test set for the final evaluation.


In [ ]:
# Define machine-learning helper functions

all_runs = []   # Store experiments for comparison


def scores(model, features, df):
    y_true = df[LABEL].values
    y_pred = model.predict(df[features])
    return {
        "MAE": metrics.mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(metrics.mean_squared_error(y_true, y_pred)),
        "R2": metrics.r2_score(y_true, y_pred),
    }


def run_experiment(name, model, features):
    """Fit a model on the training set and record its training scores."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(training_df[features], training_df[LABEL])

    train = scores(model, features, training_df)
    run = {
        "name": name,
        "model": model,
        "features": features,
        "train": train,
    }
    all_runs.append(run)

    print(
        f"{name}\n"
        f"   train: MAE = {train['MAE']:.3f} eV, "
        f"RMSE = {train['RMSE']:.3f} eV, R² = {train['R2']:.3f}"
    )
    return run


def plot_predicted_vs_true(run, df, title=None):
    """If the model were perfect, all points would lie on the red line."""
    y_pred = run["model"].predict(df[run["features"]])
    plt.figure()
    plt.scatter(df[LABEL], y_pred, s=5, alpha=0.4)
    top = max(df[LABEL].max(), y_pred.max())
    plt.plot([0, top], [0, top], c="red")
    plt.xlabel("True band gap (eV)")
    plt.ylabel("Predicted band gap (eV)")
    plt.title(title or run["name"])
    plt.show()


def compare_runs(runs=None):
    """Compare training metrics for the experiments completed so far."""
    runs = runs or all_runs
    table = pd.DataFrame([
        {
            "experiment": r["name"],
            "n features": len(r["features"]),
            "train MAE": r["train"]["MAE"],
            "train RMSE": r["train"]["RMSE"],
            "train R²": r["train"]["R2"],
        }
        for r in runs
    ])
    return table.round(3)


## Train Linear regression model with one feature

Start with a single descriptor: `electroneg_dev`.


Questions:
   - How well does the model fit? Are the points close to the red line?
   - What do the training errors suggest about the model's fit?


In [ ]:
# Train the model

features_1 = ["electroneg_dev"]

experiment_1 = run_experiment("1: linear, 1 feature", LinearRegression(), features_1)
plot_predicted_vs_true(experiment_1, training_df, "Experiment 1 (training data)")


<details>
<summary>Show solutions</summary>

```python
print("Learned weight: {:.3f} eV per unit of electroneg_dev, bias: {:.3f} eV".format(
    experiment_1["model"].coef_[0], experiment_1["model"].intercept_))

print("""
A single feature explains only part of the band gap. If the points form a wide
cloud around the red line, the model's predictions are not very accurate.
This may indicate underfitting: the model is too simple to capture all the
relationships in the data.
""")
```

</details>


## Train Linear regression model with more features

Next, investigate whether adding features improves the predictions. Experiment 2 uses three descriptors, while Experiment 3 uses **all** descriptors in the dataset.


Questions:
   - Does adding features reduce the training error?
   - Is a linear model with all features enough to fit the training data well?

Remember that lower training error alone does not guarantee better performance on unseen data. We will assess consistency using cross-validation next.


In [ ]:
# Train with more than one feature

features_2 = ["electroneg_dev", "cov_radius_dev", "max_ns_valence"]
features_3 = all_features

experiment_2 = run_experiment("2: linear, 3 features", LinearRegression(), features_2)
experiment_3 = run_experiment("3: linear, all features", LinearRegression(), features_3)
plot_predicted_vs_true(experiment_3, training_df, "Experiment 3 (training data)")


<details>
<summary>Show solutions</summary>

```python
print(compare_runs().to_string(index=False))
print("""
Adding features often reduces training error because the model has more
information available. However, lower training error does not necessarily mean
better performance on unseen materials. A model can fit the training data well
but generalise poorly. Cross-validation helps us assess how consistently the
model performs on different subsets of the training data.
""")
```

</details>


## Validate Linear Regression using cross validation

So far, we have compared ordinary Linear Regression models with different feature sets using their training performance. Training scores alone cannot tell us how consistently a model will perform on unseen data.

In this exercise, we will use **5 fold cross validation on the training data**. Each fold takes a turn as the validation set, while the model is trained on the other four folds.



### Question 1: Training data

Select the features and target variable from `training_df`. Use `all_features` for the input features and `LABEL` for the target.

<details>
<summary>Show solution — Question 1</summary>

```python
tuning_features = all_features

X_train = training_df[tuning_features]
y_train = training_df[LABEL]
```

</details>


### Question 2: Set up 5 fold cross validation

Use `KFold` to create a five-fold cross-validation strategy with shuffling enabled and a random state of `0`.

Why might shuffling be useful before splitting the data into folds?

<details>
<summary>Show solution — Question 2</summary>

```python
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)
```

</details>


### Question 3: Evaluate Linear Regression using cross-validation

Use `cross_validate` to evaluate a `LinearRegression` model with the cross-validation strategy from Question 2.

Calculate the MAE, RMSE and R² for each validation fold. Also calculate the training MAE for each fold so you can compare training and validation performance.

<details>
<summary>Show solution — Question 3</summary>

```python
model = LinearRegression()

cv_scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2",
    },
    return_train_score=True,
)

cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Training MAE": -cv_scores["train_MAE"],
    "Validation MAE": -cv_scores["test_MAE"],
    "Validation RMSE": -cv_scores["test_RMSE"],
    "Validation R²": cv_scores["test_R2"],
})

display(cv_results.round(4))
```

Scikit-learn returns negative values for MAE and RMSE because its scoring convention treats higher scores as better. We reverse the signs to display the usual positive error values.

</details>


### Question 4: Evaluate test set

Calculate the mean and standard deviation of the validation MAE. Then fit Linear Regression on the full training set and evaluate it on `test_df`.

Why should the test set not be used during model selection?

<details>
<summary>Show solution — Question 4</summary>

```python
print(
    "Mean validation MAE: {:.3f} eV".format(
        cv_results["Validation MAE"].mean()
    )
)

print(
    "Standard deviation of validation MAE: {:.3f} eV".format(
        cv_results["Validation MAE"].std()
    )
)

# Fit the final model using all training data
final_model = LinearRegression()
final_model.fit(X_train, y_train)

# Evaluate once on the held-out test set
test_scores = scores(final_model, tuning_features, test_df)

print("\nFinal Linear Regression — held-out test set")
print("MAE:  {:.3f} eV".format(test_scores["MAE"]))
print("RMSE: {:.3f} eV".format(test_scores["RMSE"]))
print("R²:   {:.3f}".format(test_scores["R2"]))
```

The test set is kept separate so it can provide a final, independent desivon of the model. Repeatedly using test results to choose features or models can make that evaluation overly optimistic.

</details>
